# TileUniverse Quickstart

**GPU-accelerated parallel universe simulator** - 40B logic evals/sec on RTX 4070

This notebook demonstrates the core features of TileUniverse.

In [ ]:
import tileuniverse as tu
import numpy as np

# Check GPU availability
print(f"TileUniverse v{tu.__version__}")
print(f"GPU Available: {tu.is_cuda_available()}")
if tu.is_cuda_available():
    print(f"GPU: {tu.cuda_device_name()}")

## 1. Basic Usage - Game of Life

In [ ]:
# Create engine with 1 world, 64x64 cells, Game of Life ruleset
engine = tu.Engine(worlds=1, size=(64, 64), ruleset="gol")
print(engine)

# Initialize with random pattern (30% density)
engine.randomize(0, density=0.3)
print(f"Initial alive cells: {(engine.get_world(0) != 0).sum()}")

In [ ]:
# Visualize initial state
print("Initial state:")
print(tu.render_ascii(engine, width=64, height=32))

In [ ]:
# Evolve 10 steps
engine.evolve(10)
print(f"After 10 steps (step={engine.step}):")
print(tu.render_ascii(engine, width=64, height=32))

## 2. Parallel Worlds

In [ ]:
# Create 5 parallel worlds - they evolve independently
engine = tu.Engine(worlds=5, size=(64, 64), ruleset="gol")

# Initialize each world with different random seeds (via different density)
for w in range(engine.worlds):
    engine.randomize(w, density=0.2 + w * 0.1)  # 0.2, 0.3, 0.4, 0.5, 0.6

print(f"Created {engine.worlds} parallel worlds")

# Evolve all worlds simultaneously
engine.evolve(50)
print(f"All worlds evolved to step {engine.step}")

# Show alive counts for each world
for w in range(engine.worlds):
    alive = (engine.get_world(w) != 0).sum()
    print(f"  World {w}: {alive} alive cells")

## 3. Reversible Mode (Time Travel)

In [ ]:
# Create engine with reversible mode enabled
engine = tu.Engine(
    worlds=1, 
    size=(32, 32), 
    ruleset="gol",
    reversible=True,      # Enable history tracking
    max_history=100       # Keep last 100 states
)
print(engine)
print(f"Is reversible: {engine.is_reversible}")

In [ ]:
# Initialize and evolve
engine.randomize(0, 0.3)
engine.evolve(20)
print(f"After evolve(20): step={engine.step}, history_len={engine.history_len}")

# Save state at step 20
state_20 = engine.get_world(0).copy()
alive_20 = (state_20 != 0).sum()
print(f"  Alive at step 20: {alive_20}")

In [ ]:
# Rewind 10 steps
rewound = engine.rewind(10)
print(f"Rewound {rewound} steps, now at step {engine.step}")
print(f"  Can rewind more: {engine.can_rewind()}")
print(f"  Can forward: {engine.can_forward()}")

In [ ]:
# Fast-forward back to step 20
forwarded = engine.forward(10)
print(f"Forwarded {forwarded} steps, now at step {engine.step}")

# Verify state is exactly the same
state_restored = engine.get_world(0)
print(f"State matches step 20: {np.array_equal(state_20, state_restored)}")

## 4. Performance Benchmark

In [ ]:
# Run GPU benchmark
if tu.is_cuda_available():
    result = tu.benchmark(worlds=5, width=512, height=512, steps=1000, depth=50)
    print(result)
    print(f"\nPerformance: {result.evals_per_sec / 1e9:.1f}B evals/sec")
else:
    print("GPU not available - benchmark requires CUDA")

## 5. Different Rulesets

In [ ]:
print("Available rulesets:", tu.RULESETS)
print()

# Demo each ruleset
for ruleset in tu.RULESETS:
    e = tu.Engine(worlds=1, size=(32, 32), ruleset=ruleset)
    e.randomize(0, 0.3)
    e.evolve(5)
    alive = (e.get_world(0) != 0).sum()
    print(f"{ruleset:10s}: {alive:4d} alive cells after 5 steps")

## 6. Numpy Integration

In [ ]:
engine = tu.Engine(worlds=1, size=(64, 64), ruleset="gol")

# Create a custom pattern with numpy
pattern = np.zeros((64, 64), dtype=np.uint64)

# Draw a glider
glider = [
    [0, 1, 0],
    [0, 0, 1],
    [1, 1, 1]
]
for y, row in enumerate(glider):
    for x, cell in enumerate(row):
        if cell:
            pattern[10 + y, 10 + x] = 0xFFFFFFFFFFFFFFFF

# Set the world from numpy array
engine.set_world(0, pattern)

print("Initial glider:")
print(tu.render_ascii(engine, width=32, height=20))

In [ ]:
# Watch glider move (10 steps)
for i in range(10):
    engine.evolve(5)
    print(f"\n--- Step {engine.step} ---")
    print(tu.render_ascii(engine, width=32, height=20))

## 7. Config Files

In [ ]:
# Load engine from config file
engine = tu.load_config("gol_benchmark.yaml")
print(f"Loaded from config: {engine}")

---

## Summary

TileUniverse provides:

- **40B+ evals/sec** GPU-accelerated simulation
- **Parallel worlds** for RL training environments
- **Reversible mode** for debugging and analysis
- **Multiple rulesets** (GoL, Rule110, Wire, Logic)
- **Numpy integration** for data science workflows
- **Config files** for reproducible experiments